# Pré-processamento - Dataset SEER Clínico
## Tech Challenge Fase 1 - Saúde e Segurança da Mulher
**FIAP POS TECH - IADT**

**Responsável:** Natalia Cabrera (Desenvolvedora)

### Objetivo
Pipeline completo de pré-processamento do Dataset Principal Clínico (SEER),
preparando os dados para modelagem de Machine Learning.

### Etapas:
1. Carregamento e limpeza
2. Tratamento de valores ausentes
3. Encoding de variáveis categóricas (ordinal e nominal)
4. Feature engineering
5. Normalização/Padronização
6. Split treino/teste
7. Exportação dos dados processados

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
import os
import sys

# Adicionar src ao path
sys.path.insert(0, '../../src')
from config import SEER_PARAMS, MODELING, PROCESSED_DIR, FIGURES_DIR

print('Bibliotecas e configurações carregadas!')

## 1. Carregamento dos Dados

In [ ]:
# Carregar dataset SEER
df = pd.read_csv('../../data/raw/DatasetPrincipalClinico.csv',
                 sep=';', encoding='latin-1')

print(f'Dataset original: {df.shape}')
print(f'Colunas: {list(df.columns)}')
df.head(3)

## 2. Limpeza Inicial

In [ ]:
# Remover coluna completamente nula
df = df.drop(columns=['Column1'])
print(f'Removida Column1 (100% nula)')

# Remover colunas em português (redundantes - são traduções)
portuguese_cols = ['Status do Paciente', 'Raça', 'Estrogênio', 
                   'Progesterona', 'Faixa Etária', 'Estado Civil']
df = df.drop(columns=[c for c in portuguese_cols if c in df.columns])
print(f'Removidas colunas em português (redundantes): {portuguese_cols}')

print(f'\nDataset após limpeza: {df.shape}')
print(f'Colunas restantes: {list(df.columns)}')
print(f'\nValores nulos: {df.isnull().sum().sum()}')
print(f'Duplicatas: {df.duplicated().sum()}')

## 3. Criação da Variável Target

In [ ]:
# Criar target binário: 1 = Alive, 0 = Dead
df['target'] = df['Status'].map({'Alive': 1, 'Dead': 0})

print('Target criado:')
print(f'  1 (Alive/Viva): {(df["target"] == 1).sum()} ({(df["target"] == 1).mean()*100:.1f}%)')
print(f'  0 (Dead/Óbito): {(df["target"] == 0).sum()} ({(df["target"] == 0).mean()*100:.1f}%)')
print(f'\nDesbalanceamento: {(df["target"] == 1).sum() / (df["target"] == 0).sum():.1f}:1')

## 4. Encoding de Variáveis Categóricas

### 4.1 Encoding Ordinal
Variáveis com ordem natural (estágios, graus):

In [ ]:
# Encoding ordinal para variáveis com ordem natural

# T Stage: T1 < T2 < T3 < T4
t_stage_map = {'T1': 1, 'T2': 2, 'T3': 3, 'T4': 4}
df['T Stage_encoded'] = df['T Stage'].map(t_stage_map)

# N Stage: N1 < N2 < N3
n_stage_map = {'N1': 1, 'N2': 2, 'N3': 3}
df['N Stage_encoded'] = df['N Stage'].map(n_stage_map)

# 6th Stage: IIA < IIB < IIIA < IIIB < IIIC
stage_6th_map = {'IIA': 1, 'IIB': 2, 'IIIA': 3, 'IIIB': 4, 'IIIC': 5}
df['6th Stage_encoded'] = df['6th Stage'].map(stage_6th_map)

# Grade: I < II < III < IV
grade_map = {
    'Well differentiated; Grade I': 1,
    'Moderately differentiated; Grade II': 2,
    'Poorly differentiated; Grade III': 3,
    'Undifferentiated; anaplastic; Grade IV': 4
}
df['Grade_encoded'] = df['Grade'].map(grade_map)

# A Stage: Regional < Distant
a_stage_map = {'Regional': 0, 'Distant': 1}
df['A Stage_encoded'] = df['A Stage'].map(a_stage_map)

print('Encoding ordinal aplicado:')
for col, mapping in [('T Stage', t_stage_map), ('N Stage', n_stage_map),
                      ('6th Stage', stage_6th_map), ('Grade', grade_map),
                      ('A Stage', a_stage_map)]:
    print(f'  {col}: {mapping}')

### 4.2 Encoding Binário (Status Hormonal)

In [ ]:
# Encoding binário para receptores hormonais
binary_map = {'Positive': 1, 'Negative': 0}

df['Estrogen_encoded'] = df['Estrogen Status'].map(binary_map)
df['Progesterone_encoded'] = df['Progesterone Status'].map(binary_map)

print('Encoding binário aplicado:')
print(f'  Estrogen Status: Positive=1, Negative=0')
print(f'  Progesterone Status: Positive=1, Negative=0')

### 4.3 Encoding Nominal (Raça e Estado Civil)

In [ ]:
# Encoding para variáveis nominais (sem ordem)
le_race = LabelEncoder()
df['Race_encoded'] = le_race.fit_transform(df['Race'])
print(f'Race encoding: {dict(zip(le_race.classes_, le_race.transform(le_race.classes_)))}')

le_marital = LabelEncoder()
df['Marital_encoded'] = le_marital.fit_transform(df['Marital Status'])
print(f'Marital Status encoding: {dict(zip(le_marital.classes_, le_marital.transform(le_marital.classes_)))}')

## 5. Feature Engineering

In [ ]:
# Criar features derivadas

# Razão de linfonodos positivos/examinados
df['Node_Ratio'] = df['Reginal Node Positive'] / df['Regional Node Examined'].replace(0, 1)
df['Node_Ratio'] = df['Node_Ratio'].clip(0, 1)  # Garantir entre 0 e 1

# Receptor hormonal duplo
df['Double_Positive'] = ((df['Estrogen_encoded'] == 1) & (df['Progesterone_encoded'] == 1)).astype(int)
df['Double_Negative'] = ((df['Estrogen_encoded'] == 0) & (df['Progesterone_encoded'] == 0)).astype(int)

# Faixa de tamanho do tumor
df['Tumor_Size_Category'] = pd.cut(df['Tumor Size'], 
                                    bins=[0, 20, 50, 200],
                                    labels=[0, 1, 2])  # Pequeno, Médio, Grande
df['Tumor_Size_Category'] = df['Tumor_Size_Category'].astype(int)

print('Features criadas:')
print(f'  Node_Ratio: razão linfonodos positivos/examinados')
print(f'  Double_Positive: ER+ e PR+ simultâneo')
print(f'  Double_Negative: ER- e PR- simultâneo')
print(f'  Tumor_Size_Category: 0=pequeno(<20mm), 1=médio(20-50mm), 2=grande(>50mm)')
print(f'\nNovas features - estatísticas:')
print(df[['Node_Ratio', 'Double_Positive', 'Double_Negative', 'Tumor_Size_Category']].describe())

## 6. Seleção de Features para Modelagem

In [ ]:
# Selecionar features finais (encoded + numéricas + engineered)
feature_cols = [
    # Numéricas originais
    'Age', 'Tumor Size', 'Regional Node Examined', 'Reginal Node Positive', 'Survival Months',
    # Encoded ordinais
    'T Stage_encoded', 'N Stage_encoded', '6th Stage_encoded', 'Grade_encoded', 'A Stage_encoded',
    # Encoded binários
    'Estrogen_encoded', 'Progesterone_encoded',
    # Encoded nominais
    'Race_encoded', 'Marital_encoded',
    # Feature engineering
    'Node_Ratio', 'Double_Positive', 'Double_Negative', 'Tumor_Size_Category'
]

X = df[feature_cols].copy()
y = df['target'].copy()

print(f'Features selecionadas: {len(feature_cols)}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'\nFeatures:')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

# Verificar nulos
print(f'\nValores nulos em X: {X.isnull().sum().sum()}')

## 7. Padronização (StandardScaler)

In [ ]:
# Padronizar features
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print('Features padronizadas com StandardScaler')
print(f'\nMédia das features (deve ser ~0): {X_scaled.mean().mean():.6f}')
print(f'Std das features (deve ser ~1): {X_scaled.std().mean():.4f}')
print(f'\nAmostra do X_scaled:')
X_scaled.head()

## 8. Split Treino/Teste

In [ ]:
# Separar em treino e teste com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=MODELING['test_size'],
    random_state=MODELING['random_state'],
    stratify=y
)

print(f'Dados separados (estratificados por target):')
print(f'  Treino: {X_train.shape[0]} amostras ({(1-MODELING["test_size"])*100:.0f}%)')
print(f'  Teste:  {X_test.shape[0]} amostras ({MODELING["test_size"]*100:.0f}%)')
print(f'\n  Distribuição no treino:')
print(f'    Alive: {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)')
print(f'    Dead:  {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)')
print(f'\n  Distribuição no teste:')
print(f'    Alive: {(y_test==1).sum()} ({(y_test==1).mean()*100:.1f}%)')
print(f'    Dead:  {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%)')

## 9. Exportar Dados Processados

In [ ]:
# Salvar datasets processados
os.makedirs('../../data/processed', exist_ok=True)

X_train.to_csv('../../data/processed/seer_X_train.csv', index=False)
X_test.to_csv('../../data/processed/seer_X_test.csv', index=False)
y_train.to_csv('../../data/processed/seer_y_train.csv', index=False)
y_test.to_csv('../../data/processed/seer_y_test.csv', index=False)

# Salvar scaler
import joblib
os.makedirs('../../models', exist_ok=True)
joblib.dump(scaler, '../../models/seer_scaler.pkl')

# Salvar nomes das features
pd.Series(feature_cols).to_csv('../../data/processed/seer_feature_names.csv', index=False)

print('Dados processados salvos em data/processed/')
print(f'  seer_X_train.csv: {X_train.shape}')
print(f'  seer_X_test.csv: {X_test.shape}')
print(f'  seer_y_train.csv: {y_train.shape}')
print(f'  seer_y_test.csv: {y_test.shape}')
print(f'  seer_scaler.pkl: StandardScaler salvo')
print(f'  seer_feature_names.csv: {len(feature_cols)} features')

## 10. Visualização Final: Distribuição das Features Processadas

In [ ]:
# Visualizar distribuições após padronização
fig, axes = plt.subplots(3, 6, figsize=(24, 14))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].hist(X_scaled[col], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].axvline(0, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Distribuição das Features Após Padronização', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../../reports/figures/seer_features_scaled.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Pipeline de pré-processamento concluído com sucesso!')
print(f'\nResumo: {df.shape[0]} registros → {len(feature_cols)} features → Split {int((1-MODELING["test_size"])*100)}/{int(MODELING["test_size"]*100)}')

## Resumo do Pipeline

| Etapa | Ação | Resultado |
|-------|------|----------|
| 1. Limpeza | Remover Column1 + colunas PT | 15 colunas úteis |
| 2. Target | Status → binário (Alive=1, Dead=0) | Desbalanceado 5.5:1 |
| 3. Encoding Ordinal | T/N/6th Stage, Grade, A Stage | 5 features |
| 4. Encoding Binário | Estrogen, Progesterone | 2 features |
| 5. Encoding Nominal | Race, Marital Status | 2 features |
| 6. Feature Engineering | Node_Ratio, Double+/-, Tumor_Category | 4 features |
| 7. Padronização | StandardScaler (μ=0, σ=1) | 18 features padronizadas |
| 8. Split | 80/20 estratificado | 3219 treino / 805 teste |

**Total: 18 features prontas para modelagem**